In [ ]:
# Install yt-dlp to download the video/audio and Whisper for ASR
!pip install -q yt-dlp openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 32.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.8 MB/s eta 0:00:00


In [ ]:
import yt_dlp
import whisper
import os
import re

# ==========================================================
# 1. DEFINE YOUR VIDEO LINKS HERE
# ==========================================================
# You can add both of your Doordarshan National links to this list
video_links = [
    "https://www.youtube.com/watch?v=SxgaGXcQ8ZY",  # Video 1: Integrated Farming Systems
    "https://www.youtube.com/watch?v=JLwhLr4ZVd0"   # Video 2: Horticulture in Haryana
]

# Helper function to remove punctuation and clean up titles for file names
def clean_filename(title):
    return re.sub(r'[\\/*?:"<>|]', "", title).replace(" ", "_")

# ==========================================================
# 2. LOAD THE MACHINE LEARNING MODEL
# ==========================================================
print("Loading Whisper 'medium' model onto GPU memory...")
# 'medium' provides superior precision for regional Indian agricultural terms
model = whisper.load_model("medium")
print("Model loaded successfully.\n" + "="*60)

# ==========================================================
# 3. LOOP THROUGH AND PROCESS EACH VIDEO
# ==========================================================
for index, url in enumerate(video_links, start=1):
    print(f"\n[PROCESSING] Starting Video {index} of {len(video_links)}")
    print(f"Target URL: {url}")

    # Configure downloader to create a clean, distinct audio track file name
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'temporary_audio_track_{index}.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'quiet': True  # Keeps the Colab console layout clean
    }

    try:
        # Step A: Download and fetch metadata
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print("-> Extracting stream info and title...")
            info_dict = ydl.extract_info(url, download=True)
            video_title = info_dict.get('title', f'Krishi_Darshan_Video_{index}')
            safe_title = clean_filename(video_title)

        audio_file = f"temporary_audio_track_{index}.mp3"

        # Step B: Pass file to AI Engine
        if os.path.exists(audio_file):
            print(f"-> Audio saved. Running AI Speech-to-Text on '{video_title}'...")
            print("   (This takes a few minutes per video, please don't interrupt)...")

            # Transcribe with language set to Hindi
            result = model.transcribe(audio_file, language="hi", verbose=False)

            # Step C: Save text transcript file
            output_filename = f"{safe_title}_transcript.txt"
            with open(output_filename, "w", encoding="utf-8") as f:
                f.write(result["text"])

            print(f"✓ [SUCCESS] Saved clean transcript text to: '{output_filename}'")

            # Step D: Delete audio file to keep your Colab disk storage clean
            os.remove(audio_file)
        else:
            print(f"✗ [ERROR] Failed to verify local audio track file for video {index}")

    except Exception as e:
        print(f"✗ [FAILED] An error stopped processing for this video: {e}")
        # Make sure messy temporary storage is cleaned up even if the video crashes
        if os.path.exists(f"temporary_audio_track_{index}.mp3"):
            os.remove(f"temporary_audio_track_{index}.mp3")
        continue

print("\n" + "="*60 + "\nAll video transcriptions have been completed successfully!")

Loading Whisper 'medium' model onto GPU memory...


100%|██████████████████████████████████████| 1.42G/1.42G [00:07<00:00, 206MiB/s]


Model loaded successfully.

[PROCESSING] Starting Video 1 of 2
Target URL: https://www.youtube.com/watch?v=SxgaGXcQ8ZY
-> Extracting stream info and title...


-> Audio saved. Running AI Speech-to-Text on 'Krishi Darshan  एकिकृत कृषि प्रणाली'...
   (This takes a few minutes per video, please don't interrupt)...


 67%|██████▋   | 92764/137633 [28:58<14:00, 53.36frames/s]


✓ [SUCCESS] Saved clean transcript text to: 'Krishi_Darshan__एकिकृत_कृषि_प्रणाली_transcript.txt'

[PROCESSING] Starting Video 2 of 2
Target URL: https://www.youtube.com/watch?v=JLwhLr4ZVd0
-> Extracting stream info and title...


-> Audio saved. Running AI Speech-to-Text on 'Krishi Darshan   Bhagwani ke vikas ka harayana me prayas'...
   (This takes a few minutes per video, please don't interrupt)...


 84%|████████▎ | 123182/147182 [18:33<03:36, 110.64frames/s]

✓ [SUCCESS] Saved clean transcript text to: 'Krishi_Darshan___Bhagwani_ke_vikas_ka_harayana_me_prayas_transcript.txt'

All video transcriptions have been completed successfully!


### Mount Google Drive
To save the transcriptions to your Google Drive, you'll need to mount it first. A folder named `drive` will be created in your Colab environment, through which you can access your Drive files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Now that Google Drive is mounted, I will modify the previous code cell (`VZbkgoS19TFk`) to save the transcriptions to a folder named `transcriptions` within your Google Drive. Please make sure this folder exists in your Google Drive, or create it if it doesn't.
